# Non-Invasive Fetal ECG – Feature Extraction

This notebook loads the preprocessed `.npz` recordings generated by the preprocessing notebook. It inspects ICA components, estimates heart rates, selects candidate fetal and maternal components, detects R-peaks, calculates RR intervals, extracts time-domain, frequency-domain and nonlinear HRV features, and saves one feature row for each recording.

**Important:** ICA does not automatically label fetal and maternal components. The component-selection step below is a heuristic and should be manually checked using the displayed plots.


In [ ]:
# Install the required packages once if they are not already installed
!pip install numpy pandas matplotlib scipy neurokit2

# Import libraries for numerical processing, data handling and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import signal as sig
import neurokit2 as nk

# Set a fixed seed for reproducible operations
np.random.seed(0)

# Set the default plot size for ECG visualizations
plt.rcParams["figure.figsize"] = (12, 4)


## Configuration

Update `PREPROCESSED_FOLDER` so that it points to the folder containing the `.npz` files created by the preprocessing notebook.


In [ ]:
# Folder containing the preprocessed NPZ recordings
PREPROCESSED_FOLDER = Path(r"C:\Users\07dee\OneDrive\Documents\Macroscopic_physiological system\preprocessed_data")

# Number of seconds shown in diagnostic plots
PLOT_SECONDS = 10

# Plausible heart-rate limits used only for component screening
MIN_HR = 40
MAX_HR = 220

# Display the files found in the selected folder
npz_files = sorted(PREPROCESSED_FOLDER.glob("*.npz"))
print("Number of NPZ files found:", len(npz_files))
print("First files:", [p.name for p in npz_files[:5]])


## Load and inspect one preprocessed recording

The preprocessing notebook stores the normalized abdominal signals, ICA sources and sampling frequency. This cell checks the array shapes before feature extraction.


In [ ]:
# Stop with a clear message if no preprocessed recordings were found
if not npz_files:
    raise FileNotFoundError("No .npz files found. Check PREPROCESSED_FOLDER.")

# Select the first recording for inspection
sample_file = npz_files[0]

# Load the stored arrays from the NPZ file
data = np.load(sample_file)

# Read the normalized abdominal signals, ICA sources and sampling frequency
abdomen_norm = np.asarray(data["abdomen_norm"], dtype=float)
sources = np.asarray(data["sources"], dtype=float)
fs = float(np.asarray(data["fs"]).squeeze())

# Print the main properties of the recording
print("Recording:", sample_file.name)
print("Normalized abdomen shape:", abdomen_norm.shape)
print("ICA sources shape:", sources.shape)
print("Sampling frequency:", fs, "Hz")


## Visualize the ICA components

Each ICA component is plotted separately. The plots help us inspect whether a component has a regular ECG-like pattern or appears to contain noise or artifacts.


In [ ]:
# Create a time axis for the ICA signals
time = np.arange(sources.shape[1]) / fs

# Limit the plot to the configured number of seconds
plot_samples = min(len(time), int(PLOT_SECONDS * fs))

# Create one subplot for every ICA component
fig, axes = plt.subplots(sources.shape[0], 1, figsize=(12, 2.5 * sources.shape[0]), sharex=True)
axes = np.atleast_1d(axes)

# Plot each ICA component
for i, ax in enumerate(axes):
    ax.plot(time[:plot_samples], sources[i, :plot_samples])
    ax.set_title(f"ICA Component {i + 1}")
    ax.set_ylabel("Amplitude")

# Label the common time axis
axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()


## Helper functions for signal cleaning and R-peak detection

The function below detects R-peaks using NeuroKit2. It also removes invalid values and rejects RR intervals outside the configured heart-rate range. If a component does not contain enough valid peaks, the function returns an empty result instead of stopping the complete pipeline.


In [ ]:
def detect_valid_rpeaks(ecg_signal, fs, min_hr=40, max_hr=220):
    """Detect R-peaks and return valid peak indices and RR intervals in seconds."""

    # Convert the input signal to a one-dimensional floating-point array
    signal = np.asarray(ecg_signal, dtype=float).squeeze()

    # Return empty results for invalid input signals
    if signal.ndim != 1 or len(signal) < int(fs * 2):
        return np.array([], dtype=int), np.array([], dtype=float)

    # Replace non-finite values using linear interpolation when possible
    if not np.isfinite(signal).all():
        valid = np.isfinite(signal)
        if valid.sum() < 3:
            return np.array([], dtype=int), np.array([], dtype=float)
        signal = np.interp(np.arange(len(signal)), np.flatnonzero(valid), signal[valid])

    try:
        # Clean the ECG signal before peak detection
        cleaned = nk.ecg_clean(signal, sampling_rate=fs, method="neurokit")

        # Detect ECG R-peaks; the information dictionary contains peak indices
        _, info = nk.ecg_peaks(cleaned, sampling_rate=fs, correct_artifacts=True)
        peaks = np.asarray(info.get("ECG_R_Peaks", []), dtype=int)
    except Exception:
        # Return empty arrays if NeuroKit2 cannot process this component
        return np.array([], dtype=int), np.array([], dtype=float)

    # At least three peaks are required to calculate reliable RR intervals
    if len(peaks) < 3:
        return peaks, np.array([], dtype=float)

    # Convert consecutive peak differences from samples to seconds
    rr = np.diff(peaks) / float(fs)

    # Keep only RR intervals corresponding to the allowed heart-rate range
    valid_rr = (rr >= 60.0 / max_hr) & (rr <= 60.0 / min_hr)
    rr = rr[valid_rr]

    # Return the detected peaks and the physiologically screened RR intervals
    return peaks, rr


def estimate_component_hr(ecg_signal, fs):
    """Estimate a component's median heart rate in beats per minute."""

    # Detect valid peaks and RR intervals for the component
    peaks, rr = detect_valid_rpeaks(ecg_signal, fs, MIN_HR, MAX_HR)

    # Return NaN when too few valid intervals are available
    if len(rr) < 2:
        return np.nan, len(peaks), peaks, rr

    # Convert the median RR interval into beats per minute
    heart_rate = 60.0 / np.median(rr)
    return heart_rate, len(peaks), peaks, rr


## Estimate the heart rate of every ICA component

The component with a maternal-like rate is usually near the adult heart-rate range, while a fetal-like component may have a higher rate. This is only a screening heuristic because heart rates can overlap and ICA component order is arbitrary.


In [ ]:
# Create a list to store the screening results for every component
component_results = []

# Estimate the heart rate of each ICA component
for component_index in range(sources.shape[0]):
    hr, peak_count, peaks, rr = estimate_component_hr(sources[component_index], fs)
    component_results.append({
        "component_index": component_index,
        "estimated_hr": hr,
        "peak_count": peak_count,
        "rr_count": len(rr)
    })

# Convert the screening results into a table
component_table = pd.DataFrame(component_results)
component_table


## Select candidate fetal and maternal components

The selection rule first looks for a component near the expected fetal range and a component near the expected maternal range. If no component falls in those ranges, the closest available valid heart rates are used. The result must be manually checked using the plots.


In [ ]:
# Minimum number of valid RR intervals a component must have before it is even
# considered a candidate. This is the key fix: a component with only 1-2 valid RR
# intervals can have a "median HR" that coincidentally lands close to a target
# value (140 or 75 bpm) purely by chance, even though it is mostly noise. Requiring
# a minimum RR count first prevents low-quality/noise components from ever winning
# just because of a lucky-looking median heart rate.
MIN_RR_COUNT_FOR_CANDIDATE = 30


def choose_components(component_table):
    """Choose candidate fetal and maternal component indices using HR screening.

    Components are first filtered to require a minimum number of valid RR
    intervals (MIN_RR_COUNT_FOR_CANDIDATE), so that a noisy component with only
    a handful of spurious peaks cannot be selected just because its (unreliable)
    median heart rate happens to be numerically close to the target value.
    Among the components that pass this quality gate, the one closest to the
    expected fetal/maternal heart rate is chosen.
    """

    # Keep only components with a valid estimated heart rate
    valid = component_table.dropna(subset=["estimated_hr"]).copy()

    # Stop if no component produced a usable heart-rate estimate
    if valid.empty:
        return None, None

    # Quality gate: require enough valid RR intervals before trusting a component's
    # heart-rate estimate at all. Components that fail this are only used as a
    # last-resort fallback further below.
    reliable = valid[valid["rr_count"] >= MIN_RR_COUNT_FOR_CANDIDATE]

    # Prefer fetal-like heart rates between 110 and 180 bpm, among reliable components only
    fetal_candidates = reliable[(reliable["estimated_hr"] >= 110) & (reliable["estimated_hr"] <= 180)]

    # Prefer maternal-like heart rates between 50 and 110 bpm, among reliable components only
    maternal_candidates = reliable[(reliable["estimated_hr"] >= 50) & (reliable["estimated_hr"] <= 110)]

    # Select the fetal candidate closest to 140 bpm when available
    fetal_idx = int(fetal_candidates.iloc[(fetal_candidates["estimated_hr"] - 140).abs().argmin()]["component_index"]) if not fetal_candidates.empty else None

    # Select the maternal candidate closest to 75 bpm, excluding the fetal component,
    # and preferring the candidate with the most valid RR intervals when several are
    # similarly close to 75 bpm
    maternal_candidates = maternal_candidates[maternal_candidates["component_index"] != fetal_idx]
    if not maternal_candidates.empty:
        maternal_candidates = maternal_candidates.copy()
        maternal_candidates["hr_distance"] = (maternal_candidates["estimated_hr"] - 75).abs()
        maternal_idx = int(maternal_candidates.sort_values(["hr_distance", "rr_count"], ascending=[True, False]).iloc[0]["component_index"])
    else:
        maternal_idx = None

    # Fallback: if the quality gate left no usable candidate, fall back to the full
    # (unfiltered) valid set so the pipeline still returns a best-effort answer,
    # but this recording should be flagged for manual review (see the QC cells below).
    if fetal_idx is None:
        remaining = valid[valid["component_index"] != maternal_idx]
        if not remaining.empty:
            remaining = remaining.copy()
            remaining["hr_distance"] = (remaining["estimated_hr"] - 140).abs()
            fetal_idx = int(remaining.sort_values(["hr_distance", "rr_count"], ascending=[True, False]).iloc[0]["component_index"])

    if maternal_idx is None:
        remaining = valid[valid["component_index"] != fetal_idx]
        if not remaining.empty:
            remaining = remaining.copy()
            remaining["hr_distance"] = (remaining["estimated_hr"] - 75).abs()
            maternal_idx = int(remaining.sort_values(["hr_distance", "rr_count"], ascending=[True, False]).iloc[0]["component_index"])

    return fetal_idx, maternal_idx

fetal_component, maternal_component = choose_components(component_table)
print("Candidate fetal component:", fetal_component)
print("Candidate maternal component:", maternal_component)


## Plot the selected fetal and maternal candidates

This plot is a required quality-control step. If the selected components do not look ECG-like, change the component indices manually before continuing.


In [ ]:
# Check that both candidate components were found
if fetal_component is None or maternal_component is None:
    raise RuntimeError("Could not identify both candidate components. Inspect the component table and plots manually.")

# Create a common time axis for the selected signals
time = np.arange(sources.shape[1]) / fs
plot_samples = min(len(time), int(PLOT_SECONDS * fs))

# Plot the candidate fetal and maternal signals
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(time[:plot_samples], sources[fetal_component, :plot_samples])
axes[0].set_title(f"Candidate Fetal ECG – ICA Component {fetal_component + 1}")
axes[0].set_ylabel("Amplitude")

axes[1].plot(time[:plot_samples], sources[maternal_component, :plot_samples])
axes[1].set_title(f"Candidate Maternal ECG – ICA Component {maternal_component + 1}")
axes[1].set_ylabel("Amplitude")
axes[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()


## Feature-extraction helper functions

The following functions calculate the requested features directly from RR intervals. Time-domain features are reported in milliseconds. Mean heart rate is reported in beats per minute. Frequency-domain features are calculated from an interpolated tachogram and should be interpreted cautiously for short recordings.


In [ ]:
def safe_sample_entropy(rr_seconds):
    """Calculate sample entropy and return NaN if it cannot be estimated."""
    try:
        if len(rr_seconds) < 10:
            return np.nan
        result = nk.entropy_sample(np.asarray(rr_seconds, dtype=float))
        # neurokit2's entropy_sample returns a (value, info_dict) tuple in current versions
        if isinstance(result, tuple):
            result = result[0]
        if isinstance(result, pd.DataFrame):
            return float(result.iloc[0, 0])
        if isinstance(result, pd.Series):
            return float(result.iloc[0])
        return float(np.asarray(result).squeeze())
    except Exception:
        return np.nan


def time_features(rr_seconds):
    """Return mean RR, SDNN, RMSSD and pNN50 from RR intervals."""
    rr_ms = np.asarray(rr_seconds, dtype=float) * 1000.0

    # Return missing values when there are not enough RR intervals
    if len(rr_ms) < 2:
        return {"mean_rr_ms": np.nan, "SDNN_ms": np.nan, "RMSSD_ms": np.nan, "pNN50_percent": np.nan}

    # Calculate the difference between successive RR intervals
    successive_differences = np.diff(rr_ms)

    # Calculate standard deviation of RR intervals
    sdnn = np.std(rr_ms, ddof=1) if len(rr_ms) > 1 else np.nan

    # Calculate root mean square of successive differences
    rmssd = np.sqrt(np.mean(successive_differences ** 2)) if len(successive_differences) else np.nan

    # Calculate the percentage of successive differences greater than 50 ms
    pnn50 = 100.0 * np.mean(np.abs(successive_differences) > 50.0) if len(successive_differences) else np.nan

    return {
        "mean_rr_ms": float(np.mean(rr_ms)),
        "SDNN_ms": float(sdnn),
        "RMSSD_ms": float(rmssd),
        "pNN50_percent": float(pnn50)
    }


def poincare_features(rr_seconds):
    """Calculate Poincare SD1 and SD2 from RR intervals."""
    rr_ms = np.asarray(rr_seconds, dtype=float) * 1000.0
    if len(rr_ms) < 3:
        return {"SD1_ms": np.nan, "SD2_ms": np.nan}

    x1 = rr_ms[:-1]
    x2 = rr_ms[1:]
    sd1 = np.sqrt(0.5) * np.std(x2 - x1, ddof=1)
    sd2 = np.sqrt(2 * np.std(rr_ms, ddof=1) ** 2 - 0.5 * np.std(x2 - x1, ddof=1) ** 2)
    return {"SD1_ms": float(sd1), "SD2_ms": float(sd2)}


def frequency_features(rpeaks, fs):
    """Calculate LF, HF and LF/HF using NeuroKit2, with safe failure handling."""
    try:
        # Create the peak dictionary format expected by NeuroKit2 HRV functions
        peak_info = {"ECG_R_Peaks": np.asarray(rpeaks, dtype=int)}

        # Calculate frequency-domain HRV features
        result = nk.hrv_frequency(peak_info, sampling_rate=fs, show=False)

        # Read the expected columns when available
        lf = float(result["HRV_LF"].iloc[0]) if "HRV_LF" in result.columns else np.nan
        hf = float(result["HRV_HF"].iloc[0]) if "HRV_HF" in result.columns else np.nan
        ratio = float(result["HRV_LFHF"].iloc[0]) if "HRV_LFHF" in result.columns else np.nan
        return {"LF": lf, "HF": hf, "LF_HF_ratio": ratio}
    except Exception:
        return {"LF": np.nan, "HF": np.nan, "LF_HF_ratio": np.nan}


def extract_features_from_signal(ecg_signal, fs, prefix):
    """Extract all requested features from one ECG component."""

    # Detect R-peaks and obtain screened RR intervals
    peaks, rr = detect_valid_rpeaks(ecg_signal, fs, MIN_HR, MAX_HR)

    # Start with missing values so failed recordings do not stop batch processing
    features = {
        f"mean_{prefix}_HR_bpm": np.nan,
        f"{prefix}_SDNN_ms": np.nan,
        f"{prefix}_RMSSD_ms": np.nan,
        f"{prefix}_pNN50_percent": np.nan,
        f"{prefix}_SampEn": np.nan,
        f"{prefix}_SD1_ms": np.nan,
        f"{prefix}_SD2_ms": np.nan,
        f"{prefix}_LF": np.nan,
        f"{prefix}_HF": np.nan,
        f"{prefix}_LF_HF_ratio": np.nan,
        f"{prefix}_peak_count": len(peaks),
        f"{prefix}_RR_count": len(rr)
    }

    # Return missing feature values if too few RR intervals are available
    if len(rr) < 2:
        return features, peaks, rr

    # Calculate the average heart rate from the mean RR interval
    features[f"mean_{prefix}_HR_bpm"] = 60.0 / np.mean(rr)

    # Calculate time-domain HRV features
    tf = time_features(rr)
    features[f"{prefix}_SDNN_ms"] = tf["SDNN_ms"]
    features[f"{prefix}_RMSSD_ms"] = tf["RMSSD_ms"]
    features[f"{prefix}_pNN50_percent"] = tf["pNN50_percent"]

    # Calculate sample entropy and Poincare features
    features[f"{prefix}_SampEn"] = safe_sample_entropy(rr)
    pf = poincare_features(rr)
    features[f"{prefix}_SD1_ms"] = pf["SD1_ms"]
    features[f"{prefix}_SD2_ms"] = pf["SD2_ms"]

    # Calculate frequency-domain features; these are mainly used for maternal HRV here
    ff = frequency_features(peaks, fs)
    features[f"{prefix}_LF"] = ff["LF"]
    features[f"{prefix}_HF"] = ff["HF"]
    features[f"{prefix}_LF_HF_ratio"] = ff["LF_HF_ratio"]

    return features, peaks, rr


## Extract features from the selected components of the sample recording


In [ ]:
# Extract fetal features from the selected fetal component
fetal_features, fetal_peaks, fetal_rr = extract_features_from_signal(
    sources[fetal_component], fs, "fetal"
)

# Extract maternal features from the selected maternal component
maternal_features, maternal_peaks, maternal_rr = extract_features_from_signal(
    sources[maternal_component], fs, "maternal"
)

# Combine both feature dictionaries into one row
sample_features = {
    "recording_id": sample_file.stem,
    **fetal_features,
    **maternal_features
}

# Display the extracted features for the sample recording
pd.DataFrame([sample_features]).T


## Batch feature extraction for all recordings

For batch processing, the same screening and feature-extraction procedure is applied to every `.npz` file. The selected component indices are saved in the output for transparency. Because automatic component identification is heuristic, the component choices should be reviewed for important experiments.


In [ ]:
def process_recording(npz_path):
    """Process one NPZ recording and return one feature dictionary."""

    # Load the stored preprocessing outputs
    data = np.load(npz_path)
    sources_local = np.asarray(data["sources"], dtype=float)
    fs_local = float(np.asarray(data["fs"]).squeeze())

    # Estimate the heart rate of each ICA component
    results = []
    for idx in range(sources_local.shape[0]):
        hr, peak_count, peaks, rr = estimate_component_hr(sources_local[idx], fs_local)
        results.append({
            "component_index": idx,
            "estimated_hr": hr,
            "peak_count": peak_count,
            "rr_count": len(rr)
        })
    table = pd.DataFrame(results)

    # Select candidate fetal and maternal components
    fetal_idx, maternal_idx = choose_components(table)

    # Start the output row with recording information
    row = {
        "recording_id": npz_path.stem,
        "sampling_rate_Hz": fs_local,
        "fetal_component": fetal_idx,
        "maternal_component": maternal_idx
    }

    # If component selection failed, return a row containing missing features
    if fetal_idx is None or maternal_idx is None or fetal_idx == maternal_idx:
        return row

    # Extract fetal and maternal features from their selected components
    fetal, _, _ = extract_features_from_signal(sources_local[fetal_idx], fs_local, "fetal")
    maternal, _, _ = extract_features_from_signal(sources_local[maternal_idx], fs_local, "maternal")

    # Add all extracted features to the output row
    row.update(fetal)
    row.update(maternal)

    return row

# Process every available NPZ recording and collect the feature rows
all_rows = []
for file_path in npz_files:
    try:
        row = process_recording(file_path)
        all_rows.append(row)
        print("Processed:", file_path.name)
    except Exception as error:
        print("Failed:", file_path.name, "|", error)
        all_rows.append({"recording_id": file_path.stem, "error": str(error)})

# Convert all rows into one final feature table
feature_df = pd.DataFrame(all_rows)
feature_df


## Add recording labels

This cell derives `ARR` and `NR` labels from the filename prefix. Use this only if the dataset documentation confirms that these prefixes represent the intended classes.


In [ ]:
def label_from_filename(recording_id):
    # Convert the recording name to uppercase for consistent matching
    name = str(recording_id).upper()

    # Assign the label based on the filename prefix
    if name.startswith("ARR"):
        return "ARR"
    if name.startswith("NR"):
        return "NR"
    return "unknown"

# Add a label column to the feature table
feature_df["label"] = feature_df["recording_id"].apply(label_from_filename)

# Display the final table with labels
feature_df


## Save the final feature dataset

The CSV file contains one row per recording and can be used later for model training, validation and explainability analysis.


In [ ]:
# Define the output CSV path
OUTPUT_CSV = PREPROCESSED_FOLDER / "fetal_maternal_features.csv"

# Save the feature table without the pandas index column
feature_df.to_csv(OUTPUT_CSV, index=False)

# Confirm that the file was saved successfully
print("Feature extraction completed.")
print("Saved feature file:", OUTPUT_CSV)
print("Final table shape:", feature_df.shape)


## Final quality checks

Before using the CSV for machine learning, check the number of recordings, missing values, labels, and the estimated heart rates. Manual inspection of the selected ICA components is recommended.


In [ ]:
# Display the number of recordings and features
print("Number of rows:", len(feature_df))
print("Number of columns:", len(feature_df.columns))

# Display the label distribution
if "label" in feature_df.columns:
    print("Label counts:")
    print(feature_df["label"].value_counts(dropna=False))

# Display the amount of missing data in each column
missing_values = feature_df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
display(missing_values.to_frame("missing_count"))

# Display the main heart-rate columns when they exist
rate_columns = [
    column for column in ["mean_fetal_HR_bpm", "mean_maternal_HR_bpm"]
    if column in feature_df.columns
]
if rate_columns:
    display(feature_df[["recording_id"] + rate_columns])


## Automatic plausibility check (new)

Rather than manually reading every row, this cell automatically flags recordings whose
extracted values fall outside expected physiological ranges. A flagged recording almost
always means the wrong ICA component was selected for that recording (fetal or maternal, or
both) and should be checked manually using the plotting cells earlier in this notebook,
overriding `fetal_component` / `maternal_component` for that specific file if needed.

In [ ]:
def flag_implausible_row(row):
    """Return a list of human-readable reasons a row looks physiologically implausible."""
    issues = []

    fhr = row.get("mean_fetal_HR_bpm", np.nan)
    if pd.notna(fhr) and not (100 <= fhr <= 180):
        issues.append(f"fetal HR out of range ({fhr:.1f} bpm, expected ~110-160)")

    mhr = row.get("mean_maternal_HR_bpm", np.nan)
    if pd.notna(mhr) and not (50 <= mhr <= 110):
        issues.append(f"maternal HR out of range ({mhr:.1f} bpm, expected ~60-100)")

    m_rmssd = row.get("maternal_RMSSD_ms", np.nan)
    if pd.notna(m_rmssd) and m_rmssd > 150:
        issues.append(f"maternal RMSSD implausibly high ({m_rmssd:.1f} ms, expected ~20-60)")

    m_pnn50 = row.get("maternal_pNN50_percent", np.nan)
    if pd.notna(m_pnn50) and m_pnn50 > 60:
        issues.append(f"maternal pNN50 implausibly high ({m_pnn50:.1f}%, expected well under 50%)")

    f_rmssd = row.get("fetal_RMSSD_ms", np.nan)
    if pd.notna(f_rmssd) and f_rmssd > 100:
        issues.append(f"fetal RMSSD implausibly high ({f_rmssd:.1f} ms, fetal short-term variability is normally low)")

    return issues


# Apply the check to every row in the final feature table
feature_df["quality_flags"] = feature_df.apply(flag_implausible_row, axis=1)
feature_df["needs_review"] = feature_df["quality_flags"].apply(lambda x: len(x) > 0)

flagged = feature_df[feature_df["needs_review"]]

print(f"{len(flagged)} / {len(feature_df)} recordings flagged for manual review.\n")
for _, r in flagged.iterrows():
    print(f"{r['recording_id']}:")
    for issue in r["quality_flags"]:
        print(f"   - {issue}")
    print()

## Duplicate-row check (new)

Two recordings ending up with identical extracted feature values almost always indicates a
data-handling issue upstream (e.g. the same `.npz` file processed twice under different
names, or a preprocessing bug that reused the same array for two recordings) rather than a
genuine coincidence. This cell flags any such exact duplicates so they can be traced back
and fixed rather than silently entering the final dataset.

In [ ]:
# Compare only the actual extracted feature values, not metadata columns
metadata_cols = ["recording_id", "label", "fetal_component", "maternal_component",
                  "sampling_rate_Hz", "quality_flags", "needs_review"]
feature_cols = [c for c in feature_df.columns if c not in metadata_cols]

# Find every row that shares identical feature values with at least one other row
duplicate_mask = feature_df.duplicated(subset=feature_cols, keep=False)
duplicates = feature_df[duplicate_mask].sort_values(by=feature_cols)

if duplicates.empty:
    print("No duplicate feature rows found.")
else:
    print(f"WARNING: {len(duplicates)} recordings have EXACTLY identical extracted features:\n")
    print(duplicates[["recording_id", "label"] + feature_cols[:4]])
    print("\nThis usually means these recordings were accidentally processed from the same")
    print("source file. Check the corresponding .npz files (and the original .dat/.hea pairs)")
    print("for these recording_id values before trusting this data.")